# PPO for structural optimization: train, benchmark, inspect

Run this notebook from the `src` folder. It is a thin front end:

- training is `train_cnn.train`, the same code as `python src/train_cnn.py`,
- scoring is `benchmark.benchmark`, which compares the policy with classical reference designs.

The environment, reward and network are described in the README.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import benchmark
import train_cnn

print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")

## Configuration

`TIMESTEPS` counts vector-environment steps; each one advances all environments.

In [ ]:
GRID_WIDTH = 12              # X direction / columns
GRID_HEIGHT = 8              # Y direction / rows
TARGET_DENSITY = 0.45
TIMESTEPS = 300_000
NUM_ENVS = 64
SEED = 0

config = train_cnn.make_config(
    width=GRID_WIDTH,
    height=GRID_HEIGHT,
    target_density=TARGET_DENSITY,
    timesteps=TIMESTEPS,
    num_envs=NUM_ENVS,
    seed=SEED,
    random_beams=True,       # False trains on the single beam_type below
    domain="mixed",          # "legacy", "rich" or "mixed" random problems
    beam_type=1,
    two_path=True,
    vector="async",          # "sync" steps environments in-process
    log_dir=Path("PPO_logs/scalable"),
    checkpoint_dir=Path("PPO_preTrained/scalable"),
)
{key: value for key, value in vars(config).items() if key in (
    "gamma", "gae_lambda", "rollout_steps", "minibatch_size", "update_epochs",
    "entropy_coef", "entropy_coef_final", "target_kl", "contact_weight", "max_refinement_swaps",
)}

## Train

Skip this cell and set `CHECKPOINT` by hand to evaluate an existing run.

In [ ]:
CHECKPOINT, LOG = train_cnn.train(config)

## Training log

One row per PPO update.

In [ ]:
log = np.genfromtxt(LOG, delimiter=",", names=True)
figure, axes = plt.subplots(2, 3, figsize=(16, 8))
panels = (
    ("mean_episode_return", "Mean episode return"),
    ("mean_final_log_ratio", "log10(C/C0) at episode end"),
    ("mean_episode_length", "Episode length"),
    ("explained_variance", "Critic explained variance"),
    ("approx_kl", "Approximate KL"),
    ("transitions_per_second", "Transitions per second"),
)
for axis, (column, title) in zip(axes.flat, panels):
    axis.plot(log["vector_steps"], log[column])
    axis.set(title=title, xlabel="Vector step")
    axis.grid(alpha=0.25)
figure.tight_layout()
plt.show()

last = log[-1]
print(f"Time split: environments {last['env_seconds']:.0f}s, "
      f"policy {last['policy_seconds']:.0f}s, updates {last['update_seconds']:.0f}s")

## Benchmark against reference designs

References are computed once per problem and cached in `benchmarks/references.json`. A gap of 0% means the policy matches the best design the classical pipeline found.

In [ ]:
agent = benchmark.load_agent(CHECKPOINT, GRID_WIDTH, GRID_HEIGHT)
problems = benchmark.build_problems(
    GRID_WIDTH, GRID_HEIGHT, TARGET_DENSITY,
    random_count=20, random_domain="mixed",
    reward_weights={"w_contact": config.contact_weight},
    max_refinement_swaps=config.max_refinement_swaps,
)
rows, summary = benchmark.benchmark(agent, problems)

## Designs

The best completed design of each episode on the nine built-in beams. Cyan squares are supports and red arrows are loads.

In [ ]:
for name, env in problems[:9]:
    result = benchmark.run_policy(agent, env)
    print(f"{name}: steps {result['steps']}, stopped by policy: {result['stopped']}")
    env.render(best=True)